# Great Expectations Basics on Databricks Free Edition

This notebook introduces **Great Expectations (GX Core)** with small, runnable PySpark examples. It creates all data in memory, so it needs no cloud storage, secrets, or paid Databricks features.

By the end, you will be able to:

- explain the main GX objects;
- test one expectation interactively;
- group rules in an Expectation Suite;
- validate good and bad Spark DataFrames; and
- stop a pipeline when critical data-quality rules fail.

> Target: Databricks Free Edition, Python notebook, GX Core 1.19.1. Attach the notebook to serverless compute and run cells from top to bottom.

## 1. Install GX Core

Databricks already supplies Spark. `%pip` installs GX only for this notebook environment. If Databricks asks you to restart Python after installation, do so and continue with the next cell.

In [ ]:
%pip install great_expectations==1.19.1

## 2. Import and check the environment

GX evaluates rules; Spark continues to hold and process the data.

In [ ]:
import great_expectations as gx
from pyspark.sql import types as T

print("GX version   :", gx.__version__)
print("Spark version:", spark.version)

## 3. The GX mental model

Think of GX as executable documentation for data quality:

| Object | Beginner meaning | This notebook |
|---|---|---|
| Data Context | GX project/session manager | an in-memory context |
| Data Source | how GX connects to a compute/data system | Spark |
| Data Asset | a named logical dataset | customer orders |
| Batch Definition | which records form one validation batch | the whole DataFrame |
| Expectation | one testable rule | `amount >= 0` |
| Expectation Suite | related rules grouped together | order quality rules |
| Validation Definition | a suite paired with a batch definition | orders + order rules |
| Checkpoint | reusable validation runner | orders quality checkpoint |

A validation does not normally clean or modify data. It measures the data and returns structured results.

## 4. Create clean and faulty sample data

Both DataFrames use the same explicit schema. The bad dataset contains a duplicate ID, a null customer, an unknown status, and a negative amount.

In [ ]:
order_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), nullable=False),
    T.StructField("customer_id", T.StringType(), nullable=True),
    T.StructField("amount", T.DoubleType(), nullable=False),
    T.StructField("status", T.StringType(), nullable=False),
])

good_rows = [
    (101, "C001", 120.50, "PAID"),
    (102, "C002", 75.00, "PENDING"),
    (103, "C003", 210.25, "PAID"),
    (104, "C004", 49.99, "CANCELLED"),
]

bad_rows = good_rows + [
    (104, None, -25.00, "UNKNOWN"),
]

good_orders_df = spark.createDataFrame(good_rows, order_schema)
bad_orders_df = spark.createDataFrame(bad_rows, order_schema)

print("Clean data")
good_orders_df.show()
print("Faulty data")
bad_orders_df.show()

## 5. Connect GX to a Spark DataFrame

An ephemeral context keeps this lesson self-contained and avoids writing project configuration to storage. The DataFrame itself is supplied later through `batch_parameters`.

In [ ]:
context = gx.get_context(mode="ephemeral")

data_source = context.data_sources.add_spark(name="spark_in_memory")
data_asset = data_source.add_dataframe_asset(name="customer_orders")
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    name="all_orders"
)

print("GX connection objects created.")

## 6. Test one Expectation interactively

An **Expectation** is a declarative assertion about data. Here, every `amount` should be between 0 and 10,000. Testing one rule on a Batch is useful while exploring data; the result is not persisted.

In [ ]:
amount_rule = gx.expectations.ExpectColumnValuesToBeBetween(
    column="amount", min_value=0, max_value=10_000
)

bad_batch = batch_definition.get_batch(
    batch_parameters={"dataframe": bad_orders_df}
)
single_result = bad_batch.validate(amount_rule, result_format="SUMMARY")

print("Passed?", single_result.success)
print("Unexpected count:", single_result.result.get("unexpected_count"))
print("Unexpected values:", single_result.result.get("partial_unexpected_list"))

## 7. Build an Expectation Suite

A suite is a reusable contract. These rules cover common quality dimensions: completeness, uniqueness, validity, volume, and schema.

`mostly=0.95` is shown on the customer rule: up to 5% nulls would be tolerated. Choose tolerances from business requirements, not by guesswork.

In [ ]:
suite = context.suites.add(
    gx.ExpectationSuite(name="order_quality_suite")
)

suite.add_expectation(
    gx.expectations.ExpectTableColumnsToMatchOrderedList(
        column_list=["order_id", "customer_id", "amount", "status"]
    )
)
suite.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(min_value=1, max_value=1_000_000)
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(column="order_id")
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id", mostly=0.95)
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="amount", min_value=0, max_value=10_000
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="status", value_set=["PAID", "PENDING", "CANCELLED"]
    )
)

print(f"Suite contains {len(suite.expectations)} expectations.")

## 8. Create a reusable Validation Definition

The Validation Definition pairs **what data to retrieve** with **which suite to apply**. At runtime, we can pass any Spark DataFrame with the expected structure.

In [ ]:
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="validate_customer_orders",
        data=batch_definition,
        suite=suite,
    )
)

## 9. Validate clean data

The overall result is successful only when every rule in the suite succeeds.

In [ ]:
good_result = validation_definition.run(
    batch_parameters={"dataframe": good_orders_df},
    result_format="SUMMARY",
)

print("Clean dataset passed:", good_result.success)
print(good_result.statistics)

## 10. Validate faulty data and read the failures

GX results are structured objects. In production, log or store this information rather than relying only on printed output.

In [ ]:
bad_result = validation_definition.run(
    batch_parameters={"dataframe": bad_orders_df},
    result_format="SUMMARY",
)

print("Faulty dataset passed:", bad_result.success)
print("Statistics:", bad_result.statistics)
print("\nFailed expectations:")

for item in bad_result.results:
    if not item.success:
        config = item.expectation_config
        print("-", config.type, config.kwargs)
        print("  unexpected_count =", item.result.get("unexpected_count"))
        print("  examples =", item.result.get("partial_unexpected_list"))

## 11. Run the suite through a Checkpoint

A Checkpoint is the orchestration layer normally called by a scheduled job or pipeline. It can group validations and later be extended with actions.

In [ ]:
checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name="orders_quality_checkpoint",
        validation_definitions=[validation_definition],
    )
)

checkpoint_result = checkpoint.run(
    batch_parameters={"dataframe": good_orders_df}
)
print("Checkpoint passed:", checkpoint_result.success)

## 12. Use validation as a pipeline quality gate

A quality gate raises an error before bad records are written to a trusted table. The demonstration catches the error so the notebook can continue.

In [ ]:
def enforce_order_quality(dataframe):
    result = validation_definition.run(
        batch_parameters={"dataframe": dataframe},
        result_format="SUMMARY",
    )
    if not result.success:
        failed = [
            item.expectation_config.type
            for item in result.results
            if not item.success
        ]
        raise ValueError(f"Data quality gate failed: {failed}")
    return dataframe

trusted_orders_df = enforce_order_quality(good_orders_df)
print("Clean data accepted; row count =", trusted_orders_df.count())

try:
    enforce_order_quality(bad_orders_df)
except ValueError as error:
    print("Faulty data rejected as expected.")
    print(error)

## 13. Beginner exercises

1. Change one clean order amount to `15000`. Which rule fails?
2. Add `REFUNDED` to the allowed status set, then rerun the suite.
3. Change `mostly=0.95` to `mostly=1.0`. What business rule does that express?
4. Add `ExpectColumnValuesToMatchRegex(column="customer_id", regex=r"^C[0-9]{3}$")`.
5. Replace the in-memory rows with a Delta table: `orders_df = spark.table("catalog.schema.orders")`, then pass it through the same gate.

## 14. Recap and practical guidance

- Start with a few high-value rules tied to business impact.
- Use exact rules for identifiers and schemas; use `mostly` only when exceptions are genuinely acceptable.
- Test expectations interactively, then place stable rules in a suite.
- Run the suite from a Validation Definition or Checkpoint.
- Treat the validation result as a pipeline decision: accept, quarantine, warn, or fail.
- This lesson uses an ephemeral context. A production project should use persistent GX configuration/result stores and operational alerting.

Official references: [GX DataFrame connection](https://docs.greatexpectations.io/docs/core/connect_to_data/dataframes/), [Expectation Suites](https://docs.greatexpectations.io/docs/core/define_expectations/organize_expectation_suites/), and [Validation Definitions](https://docs.greatexpectations.io/docs/core/run_validations/run_a_validation_definition/).